# 🖼️ Image Captioner — optional LoRA fine-tune

**You do not need this notebook for the app to work.** The app uses pre-trained BLIP out of the box. Run this only if you want a *training* story for your portfolio — a small LoRA fine-tune that adapts BLIP's caption style on a tiny dataset. Fits a **free Colab T4**.

Set runtime to GPU: *Runtime → Change runtime type → T4 GPU*.

## 1. Install

In [ ]:
!pip install -q transformers peft datasets accelerate pillow


## 2. Load a small caption dataset
We use a few hundred examples so it trains in minutes. Swap in Flickr8k or a COCO slice for more.

In [ ]:
from datasets import load_dataset

ds = load_dataset('nlphuji/flickr30k', split='test[:400]')
print(ds)


## 3. Set up BLIP + LoRA
LoRA trains tiny adapter matrices instead of the whole model, so it fits free-GPU memory.

In [ ]:
import torch
from transformers import BlipForConditionalGeneration, BlipProcessor
from peft import LoraConfig, get_peft_model

model_id = 'Salesforce/blip-image-captioning-base'
processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(model_id)

lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                  target_modules=['query', 'value'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # only a tiny % is trainable


## 4. Train
A short loop — a few hundred steps is enough to shift caption style.

In [ ]:
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).train()
opt = torch.optim.AdamW(model.parameters(), lr=5e-4)

def collate(batch):
    images = [ex['image'].convert('RGB') for ex in batch]
    caps = [ex['caption'][0] for ex in batch]
    enc = processor(images=images, text=caps, padding=True,
                    return_tensors='pt')
    enc['labels'] = enc['input_ids'].clone()
    return {k: v.to(device) for k, v in enc.items()}

loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate)

for epoch in range(2):
    for i, batch in enumerate(loader):
        opt.zero_grad()
        out = model(**batch)
        out.loss.backward()
        opt.step()
        if i % 20 == 0:
            print(f'epoch {epoch} step {i} loss {out.loss.item():.3f}')


## 5. Save the adapter
Just the small LoRA weights — a few MB. Point the app at this folder to use it.

In [ ]:
model.save_pretrained('blip-lora-adapter')
print('saved to blip-lora-adapter/')


## 6. Try it


In [ ]:
from PIL import Image
model.eval()
img = ds[0]['image'].convert('RGB')
inputs = processor(img, return_tensors='pt').to(device)
out = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(out[0], skip_special_tokens=True))


## Using the adapter in the app
In `app/app.py`, change `get_captioner()` to `get_captioner(adapter_dir='path/to/blip-lora-adapter')`.